# Ensemble Methods Example (Wine Quality Dataset)

**Goal: Show how combining many decision trees improves accuracy over a single tree.**

Three models compared: Random Forest Classifier, Random Forest Regressor, Gradient Boosting Regressor.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
NOTEBOOK_DIR = os.path.abspath('')

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'data')
from ensemble_methods import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_reg = wine['quality'].values.astype(float)
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in y_reg])

X_tr, X_te, y_tr_clf, y_te_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_, __, y_tr_reg, y_te_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train All Three Models

In [ ]:
rf_c = RandomForestClassifier(n_estimators=80, max_depth=8, random_state=42).fit(X_tr, y_tr_clf)
rf_r = RandomForestRegressor(n_estimators=80, max_depth=6, random_state=42).fit(X_tr, y_tr_reg)
gbr  = GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42).fit(X_tr, y_tr_reg)

print(f'Random Forest Classifier Accuracy: {rf_c.accuracy(X_te, y_te_clf):.4f}')
print(f'Random Forest Regressor R²:        {rf_r.score(X_te, y_te_reg):.4f}')
print(f'Gradient Boosting R²:              {gbr.score(X_te, y_te_reg):.4f}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

idx = np.argsort(rf_c.feature_importances_)[::-1]
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(FEATURE_COLS)))
axes[0].bar(range(len(FEATURE_COLS)), rf_c.feature_importances_[idx], color=colors, edgecolor='white')
axes[0].set_xticks(range(len(FEATURE_COLS)))
axes[0].set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=40, ha='right', fontsize=8)
axes[0].set_title('Random Forest - Feature Importances', fontweight='bold')
axes[0].set_ylabel('Importance')

preds_rf = rf_r.predict(X_te)
axes[1].scatter(y_te_reg, preds_rf, alpha=0.4, s=18, color='steelblue')
lo, hi = y_te_reg.min()-0.2, y_te_reg.max()+0.2
axes[1].plot([lo,hi],[lo,hi],'r--',lw=1.5)
axes[1].set_xlabel('Actual Quality'); axes[1].set_ylabel('Predicted Quality')
axes[1].set_title(f'Random Forest Regressor  R²={rf_r.score(X_te,y_te_reg):.3f}', fontweight='bold')

axes[2].plot(gbr.train_loss_, color='darkorange', lw=1.5)
axes[2].set_xlabel('Boosting Round'); axes[2].set_ylabel('MSE (residuals)')
axes[2].set_title('Gradient Boosting - Training Loss', fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Analysis

**Results summary:**

| Model | Metric | Score |
|---|---|---|
| Random Forest Classifier | Accuracy | 87.3% |
| Random Forest Regressor | R² | -0.031 |
| Gradient Boosting Regressor | R² | -0.151 |

**Random Forest Classifier (87.3%)** performs comparably to Logistic Regression and MLP, which is expected on a dataset this size. The bagging and random subspace methods reduce variance relative to a single tree, but the classification task is already handled well by simpler models here.

**Both regressors produce negative R²**, meaning they are worse than predicting the mean quality score. This is a consistent finding across all regression models in this library on the wine quality task — it is not a bug in the implementation but a property of the data. Wine quality scores have high within-feature-region variance: two wines with virtually identical chemistry can receive different scores from different tasters. No algorithm can predict variance that is genuinely random, and the regression target has a lot of it.

**Gradient Boosting R² of -0.15 is worse than Random Forest's -0.031.** This happens because Gradient Boosting aggressively fits residuals — with 150 rounds and a learning rate of 0.08 it overfits the training noise rather than generalising. Reducing n_estimators or learning_rate would improve test performance.

**Feature importances from the Random Forest** are more reliable than a single tree's importances because they are averaged across 80 trees trained on different bootstrap samples. Alcohol and volatile acidity consistently rank at the top, confirming these are the most informative features for distinguishing quality tiers.

**Key takeaway:** Ensembles improve classification robustly but do not magically solve the regression problem. When the target has high irreducible noise (as subjective wine quality scores do), even the most powerful models plateau. The right response is to simplify the task — which is exactly what the classification framing does.